## Step 1. Import libraries and define dataset paths

This cell imports the packages needed for:
- file and path handling
- data cleaning
- sequence parsing
- numerical data processing
- sparse matrix saving

In [1]:
import sys
print(sys.executable)
print(sys.version)

/global/home/users/yushanfu/esm_env/bin/python
3.11.7 (main, Dec 15 2023, 18:12:31) [GCC 11.2.0]


In [1]:
## 1. Import libraries and define file paths
#!pip install Bio


import os
import pickle
from itertools import product

import numpy as np
import pandas as pd
from Bio import SeqIO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from scipy.sparse import csr_matrix, save_npz

# Update this path if your CAFA-5 folder lives somewhere else
BASE_DIR = "cafa-5-protein-function-prediction"
TRAIN_DIR = os.path.join(BASE_DIR, "Train")

train_fasta_path = os.path.join(TRAIN_DIR, "train_sequences.fasta")
train_terms_path = os.path.join(TRAIN_DIR, "train_terms.tsv")
train_taxonomy_path = os.path.join(TRAIN_DIR, "train_taxonomy.tsv")

for p in [train_fasta_path, train_terms_path, train_taxonomy_path]:
    print(p, "->", os.path.exists(p))
## 2. Load raw data
# Load sequences from FASTA
sequences_dict = {}
duplicate_id_count = 0
missing_id_count = 0
empty_seq_count = 0

for record in SeqIO.parse(train_fasta_path, "fasta"):
    seq_id = record.id
    seq = str(record.seq).strip()

    if not seq_id:
        missing_id_count += 1
        continue

    if seq_id in sequences_dict:
        duplicate_id_count += 1

    if len(seq) == 0:
        empty_seq_count += 1

    sequences_dict[seq_id] = seq

print("Raw sequences loaded:", len(sequences_dict))
print("Duplicate FASTA IDs:", duplicate_id_count)
print("Missing IDs:", missing_id_count)
print("Empty sequences:", empty_seq_count)

# Load labels and taxonomy
terms = pd.read_csv(train_terms_path, sep="\t")
taxonomy = pd.read_csv(train_taxonomy_path, sep="\t")

print("train_terms shape:", terms.shape)
print("train_taxonomy shape:", taxonomy.shape)
terms.head()
## 3. Check missing values
print("Missing values in train_terms:")
print(terms.isnull().sum())
print()

print("Missing values in train_taxonomy:")
print(taxonomy.isnull().sum())
# Remove missing values in the label table
terms = terms.dropna(subset=["EntryID", "term"]).copy()
print("train_terms shape after dropping missing EntryID/term rows:", terms.shape)
## 4. Remove invalid sequences

# We remove:
# - empty sequences
# - sequences containing non-standard amino acids

# Standard amino acids:
# `ACDEFGHIKLMNPQRSTVWY`
valid_aas = set("ACDEFGHIKLMNPQRSTVWY")

clean_sequences_dict = {}
removed_empty_ids = []
removed_invalid = []   # (protein_id, bad_chars)

for seq_id, seq in sequences_dict.items():
    seq = seq.strip()

    if len(seq) == 0:
        removed_empty_ids.append(seq_id)
        continue

    bad = set(seq) - valid_aas
    if bad:
        removed_invalid.append((seq_id, bad))
        continue

    clean_sequences_dict[seq_id] = seq

print("Removed empty sequences:", len(removed_empty_ids))
print("Removed invalid sequences:", len(removed_invalid))
print("Remaining clean sequences:", len(clean_sequences_dict))
# Show a few removed invalid sequences
removed_invalid[:10]
# Replace the raw dictionary with the cleaned one
sequences_dict = clean_sequences_dict
## 5. Basic sequence statistics
lengths = [len(seq) for seq in sequences_dict.values()]

print("Number of proteins:", len(sequences_dict))
print("Mean length:", np.mean(lengths))
print("Median length:", np.median(lengths))
print("Min length:", np.min(lengths))
print("Max length:", np.max(lengths))
print("Std length:", np.std(lengths))

## 6. Build label dictionary and align sequences with labels
labels_dict = terms.groupby("EntryID")["term"].apply(list)

sequence_ids = set(sequences_dict.keys())
label_ids = set(labels_dict.index)

print("Proteins with sequence only:", len(sequence_ids - label_ids))
print("Proteins with label only:", len(label_ids - sequence_ids))
print("Proteins with both:", len(sequence_ids & label_ids))
protein_ids = []
sequences = []
all_labels = []

for pid, seq in sequences_dict.items():
    if pid in labels_dict:
        protein_ids.append(pid)
        sequences.append(seq)
        all_labels.append(labels_dict[pid])

print("Aligned proteins retained:", len(sequences))
## 7. Multi-label encoding
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import save_npz
import numpy as np
import pickle

mlb = MultiLabelBinarizer(sparse_output=True)
Y_sparse = mlb.fit_transform(all_labels)

print("Y_sparse shape:", Y_sparse.shape, flush=True)

save_npz("Y_sparse.npz", Y_sparse)
np.save("go_terms.npy", mlb.classes_)
np.save("protein_ids.npy", np.array(protein_ids, dtype=object))

with open("mlb.pkl", "wb") as f:
    pickle.dump(mlb, f)

print("Saved Y_sparse.npz, go_terms.npy, protein_ids.npy, mlb.pkl", flush=True)

print("Number of GO terms:", len(mlb.classes_), flush=True)

labels_per_protein = np.asarray(Y_sparse.sum(axis=1)).ravel()
label_counts = np.asarray(Y_sparse.sum(axis=0)).ravel()

used_go_terms = set()
for labels in all_labels:
    used_go_terms.update(labels)

print("Number of GO used:", len(used_go_terms), flush=True)

terms_clean_go = terms[terms["term"].isin(used_go_terms)]
aspect_dict = terms_clean_go.set_index("term")["aspect"].to_dict()

print(list(used_go_terms)[:20], flush=True)
print("Total GO after cleaning:", len(used_go_terms), flush=True)

cafa-5-protein-function-prediction/Train/train_sequences.fasta -> True
cafa-5-protein-function-prediction/Train/train_terms.tsv -> True
cafa-5-protein-function-prediction/Train/train_taxonomy.tsv -> True
Raw sequences loaded: 142246
Duplicate FASTA IDs: 0
Missing IDs: 0
Empty sequences: 0
train_terms shape: (5363863, 3)
train_taxonomy shape: (142246, 2)
Missing values in train_terms:
EntryID    0
term       0
aspect     0
dtype: int64

Missing values in train_taxonomy:
EntryID       0
taxonomyID    0
dtype: int64
train_terms shape after dropping missing EntryID/term rows: (5363863, 3)
Removed empty sequences: 0
Removed invalid sequences: 1677
Remaining clean sequences: 140569
Number of proteins: 140569
Mean length: 553.7165946972661
Median length: 412.0
Min length: 3
Max length: 35375
Std length: 639.6512065166373
Proteins with sequence only: 0
Proteins with label only: 1677
Proteins with both: 140569
Aligned proteins retained: 140569
Y_sparse shape: (140569, 31454)
Saved Y_sparse.npz,

## Step 2. Install model dependencies

This cell installs the main deep learning packages required to run ESM-2:
- `transformers` for loading the pretrained model and tokenizer
- `torch` for tensor operations and GPU execution

This step is mainly needed when the runtime environment does not already include these packages.

In [4]:
!pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable


## Step 3. Load the pretrained ESM-2 model

This cell:
1. imports PyTorch and Hugging Face model utilities,
2. selects the pretrained checkpoint `facebook/esm2_t6_8M_UR50D`,
3. detects whether a GPU is available,
4. loads the tokenizer and model,
5. moves the model to the selected device, and
6. switches the model into evaluation mode.

The printed device helps confirm whether the notebook is using **CPU** or **GPU**.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "facebook/esm2_t30_150M_UR50D"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print("Device:", device)

/global/home/users/yushanfu/esm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/global/home/users/yushanfu/esm_env/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12030). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 107/107 [00:00<00:00, 1539.99it/s]
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status    

Device: cpu


## Step 4. Define a function to convert one protein sequence into an embedding

This function takes a single amino acid sequence and:
1. tokenizes it for the ESM-2 model,
2. moves the input tensors to the selected device,
3. runs the model without gradient tracking,
4. extracts the final hidden representation,
5. applies the attention mask to ignore padding, and
6. performs mean pooling over sequence positions.

The output is a fixed-length vector representation for one protein.


In [4]:
def get_esm_embedding(seq):
    inputs = tokenizer(
        seq,
        return_tensors="pt",
        truncation=False
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    last_hidden = outputs.last_hidden_state  # (1, L, 320)
    attention_mask = inputs["attention_mask"]

    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = (last_hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1)

    embedding = (summed / counts).squeeze().cpu().numpy()

    return embedding

## Step 5. Generate embeddings for all protein sequences

This cell loops through every protein sequence, applies the embedding function, and stores the result in a list.

- `tqdm(sequences)` displays a progress bar while the embeddings are being computed.
- Each sequence is converted into one numerical feature vector.
- Finally, the list of vectors is converted into a NumPy array called `X_esm`.

This array becomes the main feature matrix used for downstream learning.


In [5]:
from tqdm import tqdm
import numpy as np

print("Start embedding...", flush=True)
esm_embeddings = []

for seq in tqdm(sequences):
    emb = get_esm_embedding(seq)
    esm_embeddings.append(emb)

X_esm = np.array(esm_embeddings)

print(X_esm.shape)

100%|██████████| 140569/140569 [1:55:55<00:00, 20.21it/s]   


(140569, 320)


## Step 6. Check feature and label dimensions

This cell prints the shapes of:
- `X_esm`: the ESM embedding feature matrix
- `Y`: the label matrix

This is a quick sanity check to confirm that the number of protein samples matches between inputs and outputs.


In [6]:
print(X_esm.shape)
print(Y.shape)

(140569, 320)
(140569, 31454)


## Step 7. Inspect one example embedding

This cell prints:
- the first 10 values of the first embedding vector,
- the sum of that vector,
- and its standard deviation.

This helps verify that the embedding values look numerical and non-trivial, rather than being all zeros or malformed.

In [7]:
print(X_esm[0][:10])
print("sum =", X_esm[0].sum())
print("std =", X_esm[0].std())

[-0.05408447 -0.10386752  0.12199136  0.09000657  0.14335941 -0.10957734
 -0.00794093 -0.15310095 -0.00577303 -0.14568625]
sum = -3.2126997
std = 0.31799027


## Step 8. Save the processed dataset as a compressed archive

This cell creates an output directory and saves the processed dataset in a compressed `.npz` file.

The saved file includes:
- `X`: the ESM feature matrix
- `Y`: the label matrix
- `ids`: the protein identifiers

This format is convenient when you want to save multiple arrays together in one file.

In [8]:
OUTPUT_DIR = "data_processed_esm"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "esm_dataset.npz"),
    X=X_esm,
    Y=Y,
    ids=protein_ids
)

In [9]:
np.save(os.path.join(OUTPUT_DIR, "X.npy"), X_esm)

In [10]:
type(Y)  # numpy.ndarray
Y.shape  # (num_proteins, num_GO_terms)

(140569, 31454)

In [11]:
np.save(os.path.join(OUTPUT_DIR, "Y.npy"), Y)